In [2]:
import pandas as pd
import numpy as np

fy2010 = "/Users/linhchung/Documents/GitHub/Cleaned-OFLC/data/raw/H-1B_FY2010.xlsx"

df = pd.read_excel(fy2010)

print(df.shape)
print(df.head())

(342575, 33)
      LCA_CASE_NUMBER               STATUS LCA_CASE_SUBMIT DECISION_DATE  \
0  I-200-09198-587210  CERTIFIED-WITHDRAWN      2009-07-20    2009-10-01   
1  I-200-09203-879743  CERTIFIED-WITHDRAWN      2009-07-22    2009-10-01   
2  I-200-09205-901931  CERTIFIED-WITHDRAWN      2009-07-24    2009-10-01   
3  I-200-09211-683464  CERTIFIED-WITHDRAWN      2009-07-30    2009-10-01   
4  I-200-09218-546645  CERTIFIED-WITHDRAWN      2009-08-06    2009-10-01   

  LCA_CASE_EMPLOYMENT_START_DATE LCA_CASE_EMPLOYMENT_END_DATE  \
0                     2009-09-30                   2012-09-30   
1                     2009-09-29                   2012-09-28   
2                     2010-01-22                   2013-01-21   
3                     2009-08-17                   2010-11-16   
4                     2009-08-10                   2012-08-10   

               LCA_CASE_EMPLOYER_NAME              LCA_CASE_EMPLOYER_ADDRESS1  \
0            XPEDITE TECHNOLOGIES INC                     

In [6]:
print(df.columns.tolist())

['LCA_CASE_NUMBER', 'STATUS', 'LCA_CASE_SUBMIT', 'DECISION_DATE', 'LCA_CASE_EMPLOYMENT_START_DATE', 'LCA_CASE_EMPLOYMENT_END_DATE', 'LCA_CASE_EMPLOYER_NAME', 'LCA_CASE_EMPLOYER_ADDRESS1', 'LCA_CASE_EMPLOYER_ADDRESS2', 'LCA_CASE_EMPLOYER_CITY', 'LCA_CASE_EMPLOYER_STATE', 'LCA_CASE_EMPLOYER_POSTAL_CODE', 'LCA_CASE_SOC_CODE', 'LCA_CASE_SOC_NAME', 'LCA_CASE_JOB_TITLE', 'LCA_CASE_WAGE_RATE_FROM', 'LCA_CASE_WAGE_RATE_TO', 'TOTAL_WORKERS', 'WORK_LOCATION_CITY1', 'WORK_LOCATION_STATE1', 'PW_1', 'PW_UNIT_1', 'PW_SOURCE_1', 'OTHER_WAGE_SOURCE_1', 'YR_SOURCE_PUB_1', 'WORK_LOCATION_CITY2', 'WORK_LOCATION_STATE2', 'PW_2', 'PW_UNIT_2', 'PW_SOURCE_2', 'OTHER_WAGE_SOURCE_2', 'YR_SOURCE_PUB_2', 'LCA_CASE_NAICS_CODE']


In [8]:
import pandas as pd

df = pd.read_excel(fy2010)

df["LCA_CASE_EMPLOYER_NAME"] = (
    df["LCA_CASE_EMPLOYER_NAME"]
    .astype("string")
    .str.strip()
    .str.upper()
)

keywords = [
    "NVIDIA",
    "APPLE",
    "MICROSOFT",
    "MICRON",
    "ADVANCED MICRO DEVICES",
    "AMAZON",
    "META",
    "FACEBOOK",
    "ALPHABET",
    "GOOGLE",
    "TESLA"
]

mask = df["LCA_CASE_EMPLOYER_NAME"].str.contains(
    "|".join(keywords),
    na=False,
    regex=True
)

print(
    df.loc[mask, "LCA_CASE_EMPLOYER_NAME"]
    .value_counts()
    .to_string()
)

LCA_CASE_EMPLOYER_NAME
MICROSOFT CORPORATION                                 4507
GOOGLE INC.                                            928
AMAZON CORPORATE LLC                                   497
APPLE INC.                                             417
MICRON TECHNOLOGY, INC.                                215
ADVANCED MICRO DEVICES, INC.                           206
FACEBOOK, INC.                                         121
NVIDIA CORPORATION                                     113
AMAZON GLOBAL RESOURCES, INC.                           99
TESLA MOTORS, INC.                                      45
APPLE INC                                               34
AMAZON FULFILLMENT SERVICES, INC.                       22
APPLEXUS TECHNOLOGIES, LLC.                             18
AMAZON CORPORTATE LLC                                   15
AMAZON WEB SERVICES LLC                                 14
METAVANTE CORPORATION                                   14
METANOIA SOLUTIONS, INC.         

In [10]:
company_map = {
    "NVIDIA CORPORATION": "NVIDIA",
    "APPLE INC.": "Apple",
    "MICROSOFT CORPORATION": "Microsoft",
    "MICRON TECHNOLOGY, INC.": "Micron",
    "ADVANCED MICRO DEVICES, INC.": "AMD",
    "AMAZON.COM, INC.": "Amazon",
    "AMAZON.COM SERVICES LLC": "Amazon",
    "META PLATFORMS, INC.": "Meta",
    "FACEBOOK, INC.": "Meta",
    "ALPHABET INC.": "Alphabet",
    "GOOGLE LLC": "Alphabet",
    "TESLA, INC.": "Tesla",
    "TESLA MOTORS, INC.": "Tesla"
}

df["Company"] = df["LCA_CASE_EMPLOYER_NAME"].map(company_map)

df = df[df["Company"].notna()].copy()

print(df["Company"].value_counts())

Company
Microsoft    4507
Apple         417
Micron        215
AMD           206
Meta          121
NVIDIA        113
Tesla          45
Name: count, dtype: int64


In [12]:
df_clean = df.copy()

In [16]:
city_names = {
    "SEATTLE": "Seattle",
    "SAN JOSE": "San Jose",
    "NEW YORK": "New York",
    "AUSTIN": "Austin",
    "DALLAS": "Dallas"
}

# Standardize city names
df["WORK_LOCATION_CITY1"] = (
    df["WORK_LOCATION_CITY1"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Create the City column
df["City"] = df["WORK_LOCATION_CITY1"].map(city_names)

# Keep only the five selected cities
df = df[df["City"].notna()].copy()

print(df["City"].value_counts())

City
Austin      120
New York     21
San Jose     11
Seattle      11
Name: count, dtype: int64


In [20]:
import pandas as pd

df["Wage_From"] = pd.to_numeric(
    df["LCA_CASE_WAGE_RATE_FROM"]
    .astype("string")
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False),
    errors="coerce"
)

print(df["Wage_From"].describe())

count           163.0
mean     84681.169018
std      20444.602377
min           44408.0
25%           67527.5
50%           81120.0
75%           93808.0
max          200087.0
Name: Wage_From, dtype: Float64


In [22]:
print(df[[
    "LCA_CASE_WAGE_RATE_FROM",
    "LCA_CASE_WAGE_RATE_TO",
    "PW_1",
    "PW_UNIT_1"
]].head(20).to_string(index=False))

 LCA_CASE_WAGE_RATE_FROM  LCA_CASE_WAGE_RATE_TO     PW_1 PW_UNIT_1
                 87503.0               142193.0  82472.0      Year
                110000.0                    NaN 107682.0      Year
                 56381.0                83036.0  56381.0      Year
                 77896.0               120318.0  77896.0      Year
                 79456.0               122321.0  79456.0      Year
                 66123.0                95469.0  66123.0      Year
                 66092.0                95469.0  66092.0      Year
                 83199.0               146730.0  83199.0      Year
                 87474.0                    NaN  81849.0      Year
                 66092.0                95469.0  66092.0      Year
                 83199.0               146730.0  83199.0      Year
                105000.0                    NaN  91957.0      Year
                 93766.0                    NaN  88234.0      Year
                 99360.0                    NaN  87693.0      

In [24]:
print(df["PW_UNIT_1"].value_counts(dropna=False))

PW_UNIT_1
Year    163
Name: count, dtype: int64


In [26]:
print(
    df.groupby("PW_UNIT_1")["Wage_From"]
    .agg(["count", "min", "median", "max"])
)

           count      min   median       max
PW_UNIT_1                                   
Year         163  44408.0  81120.0  200087.0


In [28]:
df["Annual_Wage"] = df["Wage_From"]

print(df[[
    "LCA_CASE_WAGE_RATE_FROM",
    "Annual_Wage"
]].head(10))

       LCA_CASE_WAGE_RATE_FROM  Annual_Wage
1012                   87503.0      87503.0
2476                  110000.0     110000.0
8929                   56381.0      56381.0
10570                  77896.0      77896.0
14693                  79456.0      79456.0
15879                  66123.0      66123.0
18168                  66092.0      66092.0
20487                  83199.0      83199.0
21780                  87474.0      87474.0
22277                  66092.0      66092.0


In [18]:
print("Missing wages:", df["Annual_Wage"].isna().sum())

print(df["Annual_Wage"].describe())

Missing wages: 0
count       179.000000
mean      92700.157542
std       21965.457817
min       49660.000000
25%       74500.000000
50%       89280.000000
75%      103885.000000
max      170000.000000
Name: Annual_Wage, dtype: float64


In [32]:
print(df["Annual_Wage"].describe())

print(
    df.nlargest(10, "Annual_Wage")[
        ["LCA_CASE_EMPLOYER_NAME", "Annual_Wage"]
    ]
)

count           163.0
mean     84681.169018
std      20444.602377
min           44408.0
25%           67527.5
50%           81120.0
75%           93808.0
max          200087.0
Name: Annual_Wage, dtype: Float64
              LCA_CASE_EMPLOYER_NAME  Annual_Wage
41772   ADVANCED MICRO DEVICES, INC.     200087.0
39437          MICROSOFT CORPORATION     145000.0
85013        MICRON TECHNOLOGY, INC.    140000.12
130826         MICROSOFT CORPORATION     139000.0
81246        MICRON TECHNOLOGY, INC.    133000.11
178729                    APPLE INC.     128000.0
99037          MICROSOFT CORPORATION     125250.0
219136                    APPLE INC.     125000.0
130927         MICROSOFT CORPORATION     122000.0
296023            NVIDIA CORPORATION     120910.0


In [34]:
df = df.dropna(subset=["Annual_Wage"]).copy()

df = df[df["Annual_Wage"] > 0].copy()

In [36]:
wage_summary = (
    df
    .groupby(["City", "Company"])
    .agg(
        Number_of_Records=("Annual_Wage", "count"),
        Median_Wage=("Annual_Wage", "median"),
        Mean_Wage=("Annual_Wage", "mean")
    )
    .reset_index()
)

print(wage_summary)

       City    Company  Number_of_Records  Median_Wage      Mean_Wage
0    Austin        AMD                105      71282.0    76587.47619
1    Austin      Apple                  3      96408.0       103136.0
2    Austin  Microsoft                  1      92000.0        92000.0
3    Austin     NVIDIA                 11      95000.0   97073.454545
4  New York      Apple                  1     128000.0       128000.0
5  New York       Meta                  1      44408.0        44408.0
6  New York  Microsoft                 19      98500.0  102001.421053
7  San Jose      Apple                  1      90000.0        90000.0
8  San Jose     Micron                 10    107750.05     107197.455
9   Seattle  Microsoft                 11      93500.0   92701.818182


In [40]:
columns_to_keep = [
    "Company",
    "LCA_CASE_EMPLOYER_NAME",
    "LCA_CASE_JOB_TITLE",
    "LCA_CASE_SOC_NAME",
    "City",
    "LCA_CASE_WAGE_RATE_FROM",
    "PW_UNIT_1",
    "Annual_Wage"
]

df_clean = df[columns_to_keep].copy()

df_clean.to_csv(
    "OFLC_BigTech_Wages_2010.csv",
    index=False
)